In [12]:
import random
import os
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
from concurrent.futures.process import BrokenProcessPool

In [13]:
def _simulate_chunk(args):
    trials, diff_nonbest, seed = args
    random.seed(seed)
    if diff_nonbest:
        # P1 lacks Atk, P2 lacks SpD
        p1 = [31,  0, 31, 31, 31, 31]
        p2 = [31, 31, 31, 31,  0, 31]
    else:
        # both lack Atk
        p1 = [31,  0, 31, 31, 31, 31]
        p2 = [31,  0, 31, 31, 31, 31]
    success = 0
    for _ in range(trials):
        stats = list(range(6))
        inherited = random.sample(stats, 5)
        rand_stat = next(s for s in stats if s not in inherited)
        child = [None]*6
        # Inherit 5 stats, picking a parent per stat
        for s in inherited:
            parent = random.choice([p1, p2])
            child[s] = parent[s]
        # The one random stat rolls 0..31
        child[rand_stat] = 31 if random.randint(0, 31) == 31 else 0
        if all(v == 31 for v in child):
            success += 1
    return success


In [14]:
def simulate(trials=10_000_000, diff_nonbest=True, seed=123, processes=None):
    # Determine number of processes
    if processes is None:
        processes = os.cpu_count() or 1

    # Single-process fallback for small trials or explicit request
    if processes <= 1:
        random.seed(seed)
        # stats: [HP, Atk, Def, SpA, SpD, Spe]
        if diff_nonbest:
            # P1 lacks Atk, P2 lacks SpD
            p1 = [31,  0, 31, 31, 31, 31]
            p2 = [31, 31, 31, 31,  0, 31]
        else:
            # both lack Atk
            p1 = [31,  0, 31, 31, 31, 31]
            p2 = [31,  0, 31, 31, 31, 31]
        success = 0
        for _ in range(trials):
            stats = list(range(6))
            inherited = random.sample(stats, 5)
            rand_stat = next(s for s in stats if s not in inherited)
            child = [None]*6
            # Inherit 5 stats, picking a parent per stat
            for s in inherited:
                parent = random.choice([p1, p2])
                child[s] = parent[s]
            # The one random stat rolls 0..31
            child[rand_stat] = 31 if random.randint(0, 31) == 31 else 0
            if all(v == 31 for v in child):
                success += 1
        return success / trials

    # Multiprocessing path
    # Split trials across processes as evenly as possible
    base_seed = (seed * 1000003) % (2**31 - 1)
    per_proc = trials // processes
    remainder = trials % processes
    tasks = []
    for i in range(processes):
        t = per_proc + (1 if i < remainder else 0)
        if t <= 0:
            continue
        tasks.append((t, diff_nonbest, base_seed + i + 1))

    successes = None
    # Prefer 'fork' context on POSIX to avoid pickling issues in notebooks
    try:
        ctx = mp.get_context('fork')
        with ProcessPoolExecutor(max_workers=processes, mp_context=ctx) as executor:
            successes = list(executor.map(_simulate_chunk, tasks))
    except (AttributeError, ValueError, RuntimeError, BrokenProcessPool):
        # Fallback to default context
        try:
            with ProcessPoolExecutor(max_workers=processes) as executor:
                successes = list(executor.map(_simulate_chunk, tasks))
        except BrokenProcessPool:
            # As a last resort, run single-process
            return simulate(trials, diff_nonbest, seed, processes=1)

    total_success = sum(successes)
    return total_success / trials
if __name__ == "__main__":
    print("Sim A (different non-best) ~", simulate(diff_nonbest=True))
    print("Sim B (same non-best)     ~", simulate(diff_nonbest=False))

Sim A (different non-best) ~ 0.0104412
Sim B (same non-best)     ~ 0.0052227
